### DESCRIPTION (inspired by Leetcode.com)
medium

You are given an m x n grid representing a box of oranges. Each cell in the grid can have one of three values:

"E" representing an empty cell
"F" representing a fresh orange
"R" representing a rotten orange
Every minute, any fresh orange that is adjacent (4-directionally: up, down, left, right) to a rotten orange becomes rotten.

Write a function that takes this grid as input and returns the minimum number of minutes that must elapse until no cell has a fresh orange. If it is impossible to rot every fresh orange, return -1.

Example 1:

Input:
```
grid = [
["R", "F"],
["F", "F"],
]
```
Output: 2

Explanation:

After Minute 1: The rotting orange at grid[0][0] rots the fresh oranges at grid[0][1] and grid[1][0]. After Minute 2: The rotting orange at grid[1][0] (or grid[0][1]) rots the fresh orange at grid[1][1].
So after 2 minutes, all the fresh oranges are rotten.

Example 2:

Input:
```
grid = [
["R", "E"],
["E", "F"],
]
```
Output: -1

Explanation:

The two adjacent oranges to the rotten orange at grid[0][0] are empty, so after 1 minute, there are no fresh oranges to rot. So it is impossible to rot every fresh orange.

Example 3:

Input:
```
grid = [
["R", "F", "F", "F"],
["F", "F", "F", "R"],
["E", "E", "F", "F"],
]
```
Output: 2

In [ ]:
from typing import List
from collections import deque

class Solution:
    def rotting_oranges(self, grid: List[List[str]]) -> int:
        if not grid or not grid[0]:
            return -1

        visited = set()
        queue = deque()
        rows, cols = len(grid), len(grid[0])

        for r in range(rows):
            for c in range(cols):
                if grid[r][c] == 'E':
                    visited.add((r,c))
                elif grid[r][c] == 'R':
                    visited.add((r,c))
                    queue.append((r,c))

        count = -1
        directions = [(0,-1),(-1,0),(0,1),(1,0)]

        while queue:
            count += 1
            queue_size = len(queue)
            for _ in range(queue_size):
                r, c = queue.popleft()
                for dr, dc in directions:
                    nr, nc = dr + r, dc + c
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 'F':
                        grid[nr][nc] = 'R'
                        queue.append((nr, nc))
                        visited.add((nr, nc))

        return count if len(visited) == rows * cols else -1
            

### Feedback

Your multi-source BFS and minute-level processing are correct. The bug is in the termination bookkeeping: visited tracks empty and rotten cells, but an all-empty grid should already require 0 minutes. Since the queue is empty, count remains -1, and the final check incorrectly returns -1. Track fresh oranges explicitly, or handle count == -1 when there are no fresh cells. Counting fresh cells is clearer and avoids storing visited: decrement when converting F to R, then return the elapsed minutes if the fresh count reaches zero, otherwise -1. Also ensure deque (and List, if needed) is imported. Your current approach mutates the input grid, which is usually acceptable but worth noting.


In [35]:
from typing import List
from collections import deque

class Solution:
    def rotting_oranges(self, grid: List[List[str]]) -> int:
        if not grid or not grid[0]:
            return -1

        queue = deque()
        rows, cols = len(grid), len(grid[0])
        freshCount, minutes = 0, 0
        directions = [(0,-1),(-1,0),(0,1),(1,0)]
        visited = set()
                
        for r in range(rows):
            for c in range(cols):
                if grid[r][c] == 'R':
                    queue.append((r,c))
                elif grid[r][c] == 'F':
                    freshCount += 1

        while freshCount > 0 and queue:
            minutes += 1
            queue_size = len(queue)
            for _ in range(queue_size):
                r, c = queue.popleft()
                for dr, dc in directions:
                    nr, nc = dr + r, dc + c
                    if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited and grid[nr][nc] == 'F':
                        freshCount -= 1
                        queue.append((nr, nc))
                        visited.add((nr, nc))
                        
        return minutes if freshCount == 0 else -1
            




### Feedback

Your multi-source BFS is correct and passes the tests. Processing queue_size entries per minute correctly models simultaneous rotting, and the final freshCount check handles unreachable oranges. Your visited set prevents duplicate enqueues, so the complexity is O(mn) time and O(mn) space. 

One small improvement: mark cells as visited by changing grid[nr][nc] to 'R' when enqueuing, as this avoids maintaining a separate set and makes the grid reflect the simulation. Also, visited need not track initially rotten cells because only fresh cells are enqueued. If the input contract guarantees a rectangular grid, your validation is sufficient; otherwise, uneven rows could cause indexing errors.

In [2]:
from typing import Callable

class Input:
    def __init__(self, grid: List[List[str]]):
        self.grid = grid
        
class Test:  
    def __init__(self, input: Input, result: int):
        self.input = input
        self.expected_result = result
        
def run_tests(tests: list[Test], func: Callable[[List[List[str]]], int]):
    for test in tests:
        result = func(test.input.grid)
        if result == test.expected_result:
            print(f"Test passed for {test.input.grid}")
        else:
            print(f"Test failed for {test.input.grid}. Expected: {test.expected_result}, Actual: {result}")

In [36]:
tests = [
    Test(Input([]),-1),
    Test(Input([['E']]),0),
    Test(Input([['F']]),-1),
    Test(Input([['R']]),0),
    Test(Input([['E', 'E']]),0),
    Test(Input([["R", "F"],["F", "F"]]),2),
    Test(Input([["R", "E"],["E", "F"]]),-1),
    Test(Input([["R", "F", "F", "F"],["F", "F", "F", "R"],["E", "E", "F", "F"]]),2)
]

run_tests(tests, Solution().rotting_oranges)

Test passed for []
Test passed for [['E']]
Test passed for [['F']]
Test passed for [['R']]
Test passed for [['E', 'E']]
Test passed for [['R', 'F'], ['F', 'F']]
Test passed for [['R', 'E'], ['E', 'F']]
Test passed for [['R', 'F', 'F', 'F'], ['F', 'F', 'F', 'R'], ['E', 'E', 'F', 'F']]
